# Week 2 Labs : Prompts, Structured Outputs, Function Calling

*LLMOps v2 · Week 2 · Utrains*

Week 1 got a model talking to you. Week 2 is about **controlling what it says**, so the answer is
something your software can actually use.

Every cell in this notebook does one thing: build a prompt, send it, print the answer. Run them in
order, top to bottom, and read the output before moving on. The output *is* the lesson.

| Lab | What you learn | Deck section |
|-----|----------------|--------------|
| Lab 1 | Message roles, and why the model forgets | 2 |
| Lab 2 | System prompts: vague vs specific | 3 |
| Lab 3 | Three techniques for shaping the answer | 4 |
| Lab 4 | Getting JSON your code can rely on | 5 |
| Lab 5 | Function calling, step by step by hand | 6 |
| Lab 6 | Checking whether a prompt actually works | 7 |

Everything runs on your local `llama3.2:1b` from Week 1. No cloud account, no API key, no cost.

## Where this sits in LLMOps

It is worth knowing why you are doing this before you do it.

**Week 1 was the boundary between your company and the model.** Local or hosted, which model, whose
datacentre, whose terms of service. Cost per token, latency, data sovereignty and vendor lock-in all
follow from that one decision. These labs run locally not because local is better, but because making
that choice deliberately is the skill.

**Week 2 is the boundary between the model and the rest of your software.** And here is the
difference that matters: **the consumer is not a human.** A person reads a paragraph and fills in the
gaps by themselves. A microservice cannot. `ticket.category` has to be one of five exact strings or
the router throws. Everything this week is about making the model's output safe to hand across that
boundary.

Three things make that harder than ordinary software:

- **Failure is silent.** A wrong answer does not raise. No stack trace, no 500, no alert. Your
  dashboards stay green while the model tells a customer something false. That is why you validate at
  the boundary, and why *"it looked fine when I tried it"* is not a test.
- **The same input can produce a different output.** Today, and again next month when the provider
  updates the model underneath you. So you log the model version, the prompt version and the input
  alongside the answer.
- **Quality has to be measured, not felt.** That is the *Ops* in LLMOps. Lab 6 is your first taste of
  it.

Get this wrong and it is not just an awkward answer. It is your company's reputation, and it is bad
data travelling on into every service downstream that trusted yours.

## Before you start

You need Week 1 finished: the virtual environment created, Ollama installed, and `llama3.2:1b`
pulled. If `ollama list` shows the model, you are ready.

**A note on the answers you will get.** `llama3.2:1b` is a small model. It is fast and free, which is
why we use it, but it is not always right. That is deliberate: several labs below only make sense
*because* you can watch a weak model go wrong. When your output does not exactly match what the text
describes, that is normal. Read what you actually got and ask yourself why.

---

# Lab 1 : Message roles, and why the model forgets

**Deck section 2.**

A conversation is not a piece of text. It is a **list of messages**, and each message has a **role**
that tells the model what that message is.

| Role | Who writes it | What it is for |
|------|---------------|----------------|
| `system` | You, the developer | The rules. Who the model is, what it may do, how to answer. |
| `user` | The person | The actual question. |
| `assistant` | The model | What the model said earlier in the conversation. |
| `tool` | Your code | The result of a function the model asked for. Lab 5. |

Three of them appear below. The fourth waits for Lab 5.

In [ ]:
import ollama

MODEL = "llama3.2:1b"

# A conversation is an ordinary Python list. Each message is an ordinary dict.
messages = [
    {"role": "user", "content": "What is Docker?"}
]

response = ollama.chat(model=MODEL, messages=messages)

print(response["message"]["content"])

### Add a system message

Same question. This time we put a `system` message in front of it.

The system message is the only thing that changed between this cell and the last one. Compare the two
answers.

In [ ]:
messages = [
    {"role": "system", "content": "Explain to a beginner. One short sentence, no jargon."},
    {"role": "user",   "content": "What is Docker?"},
]

response = ollama.chat(model=MODEL, messages=messages)

print(response["message"]["content"])

> **That is the whole idea of a system prompt.** You did not change the question. You changed the
> standing instructions the model reads before it answers. Lab 2 is entirely about writing those
> instructions well.

### The model has no memory

Here is the part that surprises everyone.

Ask a follow-up question, on its own, the way you would to a person who was just talking to you.

In [ ]:
messages = [
    {"role": "user", "content": "Explain that again, even simpler."}
]

response = ollama.chat(model=MODEL, messages=messages)

print(response["message"]["content"])

> **It has no idea what "that" is.**
>
> The server kept nothing from your previous call. Every call starts from zero. This is called being
> **stateless**, and it is not a bug. It is how every LLM API works, including the paid ones.

### So you send the history yourself

If you want the model to remember, **you** have to include the earlier messages. Notice the
`assistant` role appearing for the first time: that is you replaying what the model said before.

In [ ]:
messages = [
    {"role": "system",    "content": "Explain to a beginner. One short sentence, no jargon."},
    {"role": "user",      "content": "What is Docker?"},
    {"role": "assistant", "content": "Docker packs an app and all it needs into one portable box."},
    {"role": "user",      "content": "Explain that again, even simpler."},
]

response = ollama.chat(model=MODEL, messages=messages)

print(response["message"]["content"])

> Now it works. Your application is responsible for keeping the conversation and re-sending it.

### And history is not free

Re-sending the history costs something. `prompt_eval_count` tells you how many **input tokens** the
model had to read.

Run this and compare the two numbers.

In [ ]:
# The SAME question, asked two ways: alone, and with the conversation in front of it.
short_conversation = [
    {"role": "user", "content": "Explain that again, even simpler."}
]

long_conversation = [
    {"role": "system",    "content": "Explain to a beginner. One short sentence, no jargon."},
    {"role": "user",      "content": "What is Docker?"},
    {"role": "assistant", "content": "Docker packs an app and all it needs into one portable box."},
    {"role": "user",      "content": "Explain that again, even simpler."},
]

short_response = ollama.chat(model=MODEL, messages=short_conversation)
long_response = ollama.chat(model=MODEL, messages=long_conversation)

# prompt_eval_count = tokens the model READ (input).  eval_count = tokens it WROTE (output).
print("Short conversation - input:", short_response["prompt_eval_count"],
      "output:", short_response["eval_count"])
print("Long conversation  - input:", long_response["prompt_eval_count"],
      "output:", long_response["eval_count"])
print()
print("Extra input tokens, for the very same question:",
      long_response["prompt_eval_count"] - short_response["prompt_eval_count"])

> **The same question cost more, because of everything sitting in front of it.**
>
> Locally those tokens are free, so this looks like trivia. It is not. Watching this number is a real
> part of the job, and it is usually nobody else's job in the organisation.
>
> **Three things to take away.**
>
> **Input and output are billed separately, at different rates.** That is why the cell prints both.
> Output is typically the more expensive of the two, which is worth remembering the next time you are
> tempted to ask a model to think out loud (Lab 3).
>
> **The cost is per call, and history is re-sent on every call.** Turn 20 of a chat pays for turns 1
> through 19 all over again. Do the arithmetic once: a support bot handling 1,000 conversations a day
> at 20 turns each is 20,000 calls, and the average call is carrying half a conversation of history it
> already paid for. That is the line item, not the clever prompt.
>
> **Tokens are latency too.** Everything the model reads, it reads before it can answer. The same
> growth that inflates your bill is what makes turn 20 feel slower than turn 2 to the user.
>
> **What teams actually do**, in rough order of effort:
>
> - **Trim the history.** Keep only the last few turns, or replace the older ones with a short
>   summary. This is your own code, and it is the first thing to reach for.
> - **Turn on prefix caching.** When the front of your prompt is identical call after call, a long
>   system prompt for example, most providers will bill that repeated prefix at a discount. You do not
>   build this. You switch it on, and you structure the prompt so the stable part comes first and the
>   part that changes comes last.
> - **Cache the answers themselves.** If two people ask the same thing in different words, answer once
>   and serve the stored answer the second time. That one you do build, in **Week 6**.
>
> For now the habit is the point. *"Send the whole history"* is a decision with a price attached, not
> a free default, and you are the person who has to know what that price is.

---

# Lab 2 : System prompts

**Deck section 3.**

*"Be helpful"* is not an instruction, it is a hope. A good system prompt has **five parts**:

1. **Role**: who the model is.
2. **Constraints**: what it must and must not do.
3. **Output format**: how the answer should look.
4. **Examples**: what a good answer looks like.
5. **Escape hatch**: what to do when it does not know.

You will write a vague prompt and a 5-part prompt, ask each the same two questions, and compare.

### The vague prompt

This is what almost everyone writes first. It says what *not* to do, but never says how long the
answer should be, or what to do when the policy does not cover the question.

In [ ]:
POLICY = """ACME CORP LEAVE POLICY
- Parental leave: up to 20 working days per year.
- Parental leave cannot be carried forward into the next year.
- Annual leave: 25 days per year. Up to 5 unused days may be carried forward.
"""

# The f before the quotes lets us drop POLICY straight into the text, wherever {POLICY} appears.
VAGUE_PROMPT = f"""Be helpful and friendly. Answer the employee's question. Do not make things up.

{POLICY}"""

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": VAGUE_PROMPT},
        {"role": "user",   "content": "Can parental leave be carried forward?"},
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])

> **Read that answer slowly before moving on.**
>
> The model often says *"Yes, parental leave can be carried forward"* and then quotes the policy line
> saying it **cannot**, in the same breath.
>
> The correct fact is right there in its own sentence. Nothing in the prompt told it to commit to one
> short, direct answer, so it produced a plausible-sounding paragraph instead of a decision.
>
> This is why vague prompts are dangerous in production. It is not obvious nonsense you would catch
> in review. It reads perfectly well and is exactly wrong.

### Now ask it something the policy does not cover

There is nothing about bonuses in the policy. Watch what it does.

In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": VAGUE_PROMPT},
        {"role": "user",   "content": "How much is the annual bonus?"},
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])

> Usually a long, apologetic paragraph about what the policy does not contain. The honest answer is
> five words, and the prompt never told it that was allowed.

### The 5-part prompt

Same model, same policy, same questions. Only the instructions change.

Read the prompt below and find all five parts. Note there are **two** examples: one where the policy
has the answer, and one where it does not. That second example is what teaches the model *when* to
use it. Without that second example, the model refuses everything.

In [ ]:
GOOD_PROMPT = f"""You are an HR policy assistant for Acme Corp employees.

THE POLICY YOU MUST USE:
{POLICY}
RULES:
- Answer using only the policy above.
- Never state a number that is not written in the policy.

OUTPUT FORMAT:
- Reply in one or two sentences.

EXAMPLE 1 (the policy has the answer):
Question: How many days of annual leave do I get?
Answer: You get 25 days of annual leave per year.

EXAMPLE 2 (the policy does NOT have the answer):
Question: What is the company car allowance?
Answer: I cannot find that in our policies."""

print(GOOD_PROMPT)

### The same two questions, with the good prompt

In [ ]:
# Question 1: the policy answers this one.
answer_1 = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": GOOD_PROMPT},
        {"role": "user",   "content": "Can parental leave be carried forward?"},
    ],
    options={"temperature": 0},
)

# Question 2: the policy says nothing about bonuses.
answer_2 = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": GOOD_PROMPT},
        {"role": "user",   "content": "How much is the annual bonus?"},
    ],
    options={"temperature": 0},
)

print("Q1:", answer_1["message"]["content"])
print()
print("Q2:", answer_2["message"]["content"])

> **Compare the four answers you just produced.**
>
> | Question | Vague prompt | 5-part prompt |
> |---|---|---|
> | Carried forward? | often contradicts itself | short and correct |
> | Annual bonus? | a paragraph of waffle | *"I cannot find that in our policies."* |
>
> **That fifth part**, telling the model what to say when it does not know, removes more invented
> answers in production than any other line you will ever write.
>
> **An honest note.** `llama3.2:1b` is small, so it drifts: it may say *"I cannot find that
> information in our policies"* instead of the exact sentence. A better prompt improves your odds. It
> does not buy certainty, which is exactly why Lab 6 measures instead of trusting.

---

# Lab 3 : Prompt techniques

**Deck section 4.** Three techniques you will reach for constantly. Each one costs something, so each
one has a *"when it hurts"* as well as a *"when it helps"*.

## Technique 1: Few-shot, teach by example

Describing a format in words is long and easy to misread. **Showing examples** is short and exact.

First, try it with no examples at all.

In [ ]:
INSTRUCTION = """You are an IT support assistant.
Classify the ticket as DATABASE, NETWORK, APPLICATION or UNKNOWN.
"""

IT_TICKET = "The payment service cannot connect to the database."

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": INSTRUCTION},
        {"role": "user",   "content": IT_TICKET},
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])

In [ ]:
# The same instruction, but with three worked examples in front of the real question.
# The examples go in as alternating user / assistant messages.
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system",    "content": INSTRUCTION},

        {"role": "user",      "content": "My application cannot connect to the database."},
        {"role": "assistant", "content": "DATABASE"},

        {"role": "user",      "content": "Users cannot open the website."},
        {"role": "assistant", "content": "NETWORK"},

        {"role": "user",      "content": "The login button does not work."},
        {"role": "assistant", "content": "APPLICATION"},

        {"role": "user",      "content": IT_TICKET},
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])

> **Compare the two outputs.**
>
> Without examples the model usually answers with a sentence: *"The category for this ticket is
> DATABASE."* With examples it answers with one word. Neither is more *correct*, but only one of them
> can be stored in a database column without extra work. That is the value of the pattern in one line.
>
> **Now look at the ticket itself.** *"The payment service cannot connect to the database"* is
> genuinely both things: an application failing, and a database it cannot reach. Your first example
> is almost the same sentence and is labelled `DATABASE`, so the model copies it.
>
> **Try this:** change that first example's answer to `APPLICATION` and run the cell again. The
> answer flips. Your examples are not decoration. **They are the decision boundary.**
>
> **When this hurts.** Examples are tokens, and you pay for them on every call. And the model
> copies them a little too faithfully: if two of your three examples said `NETWORK`, it would start
> over-picking `NETWORK`. Notice too that `UNKNOWN` is in the instruction but has no example behind
> it. Watch whether the model ever reaches for it.

## Technique 2: Chain-of-thought, make it show its working

For problems that need several steps, asking for the answer straight away often produces a confident
wrong one. Asking the model to work through it first gives the reasoning somewhere to happen.

First, the direct question.

In [ ]:
PROBLEM = """A store sells pens at 3 for $2.10.
A customer buys 7 pens and pays with a $10 note.
How much change do they get?"""

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": f"{PROBLEM} Answer with the amount only."}],
    options={"temperature": 0},
)

print(response["message"]["content"])

In [ ]:
# The same problem, but asking the model to work through it first.
# The correct answer is $5.10.  ($2.10 / 3 = $0.70 a pen; 7 pens = $4.90; $10.00 - $4.90 = $5.10)
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": f"{PROBLEM} Think step by step, then give the final amount."}
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])

> **This is a small model, so it may get both attempts wrong.** Run each cell a few times. What you
> are looking for is not perfection, it is the *rate*: thinking step by step gets it right more often
> than answering straight away.
>
> **When this hurts.** All that reasoning is output tokens, so it costs more and takes
> longer. And your user usually does not want the working. They want the number. In production you
> let the model reason and then pull out only the final answer, which is what Lab 4 is for.
>
> Newer reasoning models already do this internally. On those, *"think step by step"* is redundant.

## Technique 3: Self-critique, review before answering

Write an answer, then check it. There are **two ways to do this**, and they are not the same tool.

- **Way A, one call.** The review checklist goes inside the prompt. The model drafts and reviews in
  one go, and you only ever see the final answer.
- **Way B, two calls.** The first call writes the answer. A second call reviews it and gives you a
  verdict you can read.

Build both, on the same task, and compare.

In [ ]:
POLICY_INFO = """Users must change their password every 90 days.
A password must contain at least 12 characters, one uppercase letter,
one lowercase letter, one number, and one special character."""

PW_QUESTION = "How often do I need to change my password?"

# WAY A: one call. The review checklist is part of the prompt.
INLINE_PROMPT = f"""You are an IT support assistant.
Answer the user's question using the information provided.

After drafting your answer, review it against these rules:
1. Does it directly answer the question?
2. Did you use only the information provided?
3. Did you miss anything important?
4. If the answer contains an error, correct it.

Return only the final answer to the user.

Information:
{POLICY_INFO}"""

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": INLINE_PROMPT},
        {"role": "user",   "content": PW_QUESTION},
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])

> Notice what you **cannot** see: whether it reviewed anything at all. You got one block of text.
>
> Now the two-call version. The first call just answers, with no mention of reviewing.

In [ ]:
# WAY B: the SAME kind of system prompt, minus the review checklist.
SYSTEM_PROMPT_PLAIN = f"""You are an IT support assistant.
Answer the user's question using the information provided.

Information:
{POLICY_INFO}"""

# Call 1: write the answer. Nothing here mentions reviewing.
draft = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_PLAIN},
        {"role": "user",   "content": PW_QUESTION},
    ],
    options={"temperature": 0},
)

draft_text = draft["message"]["content"].strip()

print("DRAFT:", draft_text)

### Way B, call 2: a fresh reviewer

This second call knows nothing about how the draft was written. It sees only the source information,
the question, and the answer.

In [ ]:
REVIEW_REQUEST = f"""You are a quality reviewer. Check the answer below.

INFORMATION:
{POLICY_INFO}

QUESTION: {PW_QUESTION}

ANSWER TO REVIEW:
{draft_text}

Does the answer use only the information above, and answer the question?
Reply with PASS or FAIL and one short reason."""

critique = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": REVIEW_REQUEST}],
    options={"temperature": 0},
)

print("VERDICT:", critique["message"]["content"].strip())

> **The verdict is a separate piece of text.** That is the whole difference. Your code can look at
> it and decide what to do next: send the answer, ask for another draft, or hand it to a human.
>
> ### Pros and cons
>
> | | **Way A, one call** | **Way B, two calls** |
> |---|---|---|
> | Cost and speed | one normal call | about twice as much |
> | Can you see the review? | no, it is invisible | yes, you can read the verdict |
> | Can your code act on it? | no | yes, PASS or FAIL is a value you can branch on |
> | Independence | none: same call, same blind spots | real: the reviewer starts fresh |
> | Usual failure | small models skip the review and just answer | the reviewer invents problems that are not there |
> | Shows up in your logs? | no | yes |
>
> **How to choose.** Use **A** when the review is a nudge and nobody downstream needs to know it
> happened. It is nearly free. Use **B** when the verdict is a **decision**: compliance checks, code
> review, "is this answer actually supported by the document". *If you cannot act on the result, you
> did not need the second call.*
>
> **When either hurts.** Both cost more, and a third pass rarely helps. Use them on the small share of
> requests where a mistake is expensive, not on everything.

---

# Lab 4 : Getting JSON your code can rely on

**Deck section 5.**

A model naturally writes prose. Your application needs **data**: a name in a name field, an email in
an email field. This lab goes up three steps, and each one is more reliable than the last.

| Step | How | What you get |
|------|-----|--------------|
| 1. Ask nicely | say "as JSON" in the prompt | *sometimes* JSON, usually wrapped in ``` fences |
| 2. JSON mode | `format="json"` | always valid JSON, but any field names it likes |
| 3. Schema | `format=` a schema | the exact fields you asked for |

The deck compares steps 2 and 3. We start at step 1, because that is where everyone actually begins
and watching it fail is the quickest way to understand the other two.

In [ ]:
RESUME = """Serge Kamgang
serge.kamgang@example.com
Cloud engineer. I work with Python, Terraform and AWS."""

EXTRACT = f"""Extract the name, email and skills as JSON. {RESUME}"""

# STEP 1: just ask for JSON and see what comes back.
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": EXTRACT}],
    options={"temperature": 0},
)

print(response["message"]["content"])

> **Look at exactly what came back.**
>
> You will usually see the JSON wrapped in a markdown code fence, and often a friendly sentence in
> front of it. A code fence is not JSON. If your program tried to read that, it would crash.
>
> People try to fix this by cutting off the fences with string tricks. Then the model adds a comment,
> or returns two objects, or uses single quotes, and the tricks grow forever. There is a proper way.

### Step 2: JSON mode

`format="json"` forces the output to be valid JSON. No fences, no friendly sentence.

What it does **not** do is control the field names.

In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": EXTRACT}],
    format="json",
    options={"temperature": 0},
)

print(response["message"]["content"])

> Always readable JSON now. But you might get `full_name` instead of `name`, or `skill` instead of
> `skills`, or an extra field you never asked for. Your code needs to know the field names in advance.

### Step 3: describe the shape you want

Here we write down the exact fields, once, as a small Python class. This is **Pydantic**, and Week 1
introduced it.

Read it as a form: three fields, and the type of each. `list[str]` means "a list of pieces of text".

In [ ]:
from pydantic import BaseModel


class Candidate(BaseModel):
    name: str
    email: str
    skills: list[str]


print("Shape defined: name, email, skills.")

In [ ]:
# model_json_schema() turns the class above into the description the model needs.
# You never write that description by hand.
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": EXTRACT}],
    format=Candidate.model_json_schema(),
    options={"temperature": 0},
)

# Check the answer really matches the shape, then use it.
candidate = Candidate.model_validate_json(response["message"]["content"])

print("name  :", candidate.name)
print("email :", candidate.email)
print("skills:", candidate.skills)

> **That is the whole point of the week in one cell.** A paragraph of text went in, and named fields
> came out: fields your code can put in a database, send to an API, or show on a page.
>
> `model_validate_json` is the important line. It **checks** the answer before you use it. If the
> model returned something that did not fit, it would raise an error here, at the boundary, instead
> of quietly writing bad data into your system.
>
> **Quick check: a question you will be asked in interviews.** *How do you make LLM output reliable?*
> Your answer: *describe the shape, make the model generate against it, and validate before using it.*
>
> **The same idea everywhere else.** Ollama takes `format=`. OpenAI calls it Structured Outputs.
> Other providers have their own name for it. The API is different every time; the idea never changes.

### Optional: the same schema, against a bigger model

Everything so far ran on your laptop. Nothing you typed left the machine.

This section changes that, and it is worth pausing on before you run it. The moment you send a prompt
to a hosted API, whatever is inside that prompt leaves your building: the resume, the policy
document, the customer's message. That is the data sovereignty decision, and here you are making it
on purpose rather than by accident.

You need an Anthropic key in a `.env` file at the repository root. Setup is in the README.
**If you do not have one, skip the next cell.** Everything else in the notebook still works.

Notice what does *not* change: the `Candidate` class. Same three fields, same validation. Only the
call wrapped around it is different.

**Why `claude-haiku-4-5` and not a bigger model?** Because of what we are asking. There are four
Claude models available, and they are priced per million tokens:

| Model | For | Input | Output |
|---|---|---|---|
| `claude-haiku-4-5` | fastest, high volume, speed-critical work | $1 | $5 |
| `claude-sonnet-5` | best balance of speed and intelligence | $2 | $10 |
| `claude-opus-5` | complex agentic coding, enterprise work | $5 | $25 |
| `claude-fable-5` | the hardest reasoning, long-running agents | $10 | $50 |

Pulling three fields out of a three-line resume is not a hard job. Running it on Opus would cost five
times as much for capability this task cannot use. **Pick the cheapest model that clears the bar, not
the best one you can afford.** How do you find out where the bar is? You measure, which is Lab 6.

In [ ]:
from dotenv import load_dotenv
import anthropic

load_dotenv()                      # reads .env from the repository root
client = anthropic.Anthropic()     # picks up ANTHROPIC_API_KEY

response = client.messages.parse(
    model="claude-haiku-4-5",
    max_tokens=512,
    messages=[{"role": "user", "content": EXTRACT}],
    output_format=Candidate,       # the SAME Pydantic class from the cell above
)

candidate = response.parsed_output

print("name  :", candidate.name)
print("email :", candidate.email)
print("skills:", candidate.skills)

> **One class, three APIs.** Ollama took `format=`. Anthropic takes `output_format=`. OpenAI takes
> `text_format=`, which is the version printed on the deck. The `Candidate` class never changed.
>
> That portability is the lesson. The shape of your data belongs to you. Which provider you send it
> to is a detail you should be able to change without rewriting your application.
>
> **Two things to compare.** The bigger model usually pulls the `skills` list more completely and gets
> the email exactly right. And this call was **billed**. A fraction of a cent on Haiku, but real. In
> one cell you crossed from free to metered and from local to remote, which is precisely the pair of
> decisions Week 1 was about.

---

# Lab 5 : Function calling, step by step

**Deck section 6.**

Sometimes the model needs something it cannot know: today's weather, a customer's order, a price
from your database. **Function calling** lets it ask your application to go and get it.

The single most important thing to understand:

> **The model never runs your code.** It sends back the *name* of a function and the *arguments* to
> use. **Your** program runs the function and hands the answer back.

There are four steps, and we are going to do each one by hand so you can see exactly what moves.

1. You send the question **plus a description of the tools available**.
2. The model replies: *"call `get_weather` with `city = Paris`"*.
3. **Your code** runs the function.
4. You send the result back, and the model writes the final answer.

### Step 1: the function, and how you describe it

Two separate things, and this is where beginners get muddled.

The **function** is ordinary Python. The model never sees this code, and it never runs it.

The **tool schema** is the description the model *does* see. It is how the model decides whether to
reach for this tool and what to pass it. Look at the `description` field: that sentence is a prompt,
and it deserves the same care as any other prompt you write.

In [ ]:
# THE FUNCTION: ordinary Python. The model never sees this code.
def get_weather(city):
    return f"It is 72F and sunny in {city}."


# THE TOOL SCHEMA: what the model DOES see, and all it has to go on.
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            },
            "required": ["city"],
        },
    },
}

print("The function works on its own :", get_weather("Paris"))
print("The model only ever sees      :", WEATHER_TOOL["function"]["description"])
print("and that it needs             :", WEATHER_TOOL["function"]["parameters"]["required"])

### Steps 2 and 3: the model asks, your code answers

The model does not answer the question. It answers with a **request**: this function, these
arguments. We write that request out by hand below so you can see its shape.

Then step 3, the part people find surprising. Nothing magic happens. You pull the argument out of the
request and call your own function with it.

In [ ]:
# STEP 2: what the model sends back. Not an answer, a request.
model_request = {
    "name": "get_weather",
    "arguments": {"city": "Paris"},
}

print("The model wants to call:", model_request["name"])
print("with the arguments     :", model_request["arguments"])

# STEP 3: YOUR code runs the function. The model never does this.
city = model_request["arguments"]["city"]
result = get_weather(city)

print()
print("Your code ran get_weather and got:", result)

> **A word of caution for later.** Those arguments came from the model, not from you. Before passing
> them to something that touches a database, real applications check them first: is this really a
> tool we have? Is the argument the right shape? You will build those checks in Week 5.

### Step 4: hand the result back

Now the fourth role finally appears. The `tool` message carries your function's answer back to the
model, and the model turns it into a sentence for the user.

If the last line gives you an error or a strange reply, do not worry. That is the small model showing
its limits, and the message list is the part that matters.

In [ ]:
# STEP 4: send the tool result back and let the model write the final answer.
messages = [
    {"role": "user",      "content": "What is the weather in Paris?"},
    {"role": "assistant", "content": "I need to call get_weather for Paris."},
    {"role": "tool",      "content": result},
    {"role": "user",      "content": "Now answer my question in one friendly sentence."},
]

print("All four roles are now in the conversation:")
print(messages[0]["role"], "->", messages[1]["role"],
      "->", messages[2]["role"], "->", messages[3]["role"])
print()

response = ollama.chat(model=MODEL, messages=messages)

print(response["message"]["content"])

> **That is the complete loop**, and every AI agent you will ever meet is built on it. The model
> decides; your application does the work; the model explains the result.
>
> **What we did by hand, real code automates.** In a real application you send `WEATHER_TOOL` along
> with the question, and the model produces step 2 on its own instead of you typing it. We skipped
> that here for two reasons: it needs more Python than this week assumes, and `llama3.2:1b` is too
> small to do it reliably. Tool calling is one of the first things a small model gets wrong.
>
> **That is a real lesson, not an excuse.** Whether a model can use tools is a question about the
> *model*, before it is a question about your code. Test it before you design around it.
>
> Week 5 builds the automatic version, with the safety checks, on a model big enough to do it.

### Optional: the real thing, with Claude

Everything above was done by hand. **You** wrote the model's request. This section lets the model
write it, which is what actually happens in production.

Same conditions as Lab 4: you need an Anthropic key, and the prompt leaves your machine.
**No key? Skip the next three cells.** The walkthrough above already taught you the loop, and the
four steps are identical.

One difference worth spotting: Anthropic describes a tool with a flat `input_schema`, not the nested
`function` wrapper you saw earlier. Every provider spells the same idea differently.

In [ ]:
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODEL_NAME = "claude-haiku-4-5"

# The same tool as before, in Anthropic's shape.
CLAUDE_WEATHER_TOOL = {
    "name": "get_weather",
    "description": "Get the current weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city name, for example Paris."}
        },
        "required": ["city"],
    },
}

QUESTION = "What is the weather in Paris?"

print("Tool offered to the model:", CLAUDE_WEATHER_TOOL["name"])

In [ ]:
# STEPS 1 and 2: send the question plus the tool. The model replies with a REQUEST, not an answer.
first = client.messages.create(
    model=MODEL_NAME,
    max_tokens=512,
    tools=[CLAUDE_WEATHER_TOOL],
    messages=[{"role": "user", "content": QUESTION}],
)

tool_use = first.content[-1]      # the last block is the tool request

print("stop_reason:", first.stop_reason)
print("block type :", tool_use.type)
print("tool name  :", tool_use.name)
print("arguments  :", tool_use.input)
print()
print("The model chose the tool and filled in the city. It did NOT answer the question.")

In [ ]:
# STEP 3: YOUR code runs the function, using the arguments the model chose.
real_result = get_weather(tool_use.input["city"])

print("Your function returned:", real_result)

# STEP 4: hand the result back and let the model write the final answer.
second = client.messages.create(
    model=MODEL_NAME,
    max_tokens=512,
    tools=[CLAUDE_WEATHER_TOOL],
    messages=[
        {"role": "user",      "content": QUESTION},
        {"role": "assistant", "content": first.content},
        {"role": "user",      "content": [{
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": real_result,
        }]},
    ],
)

print()
print("Final answer:", second.content[-1].text)

> **Same four steps, and now the model is doing its half.**
>
> Compare it with the walkthrough above. The only thing that changed is who wrote step 2. You still
> execute the function. The model still never touches your code.
>
> **Three details worth carrying forward.**
>
> `stop_reason` is `"tool_use"`. That is the signal your code checks to decide whether to run a tool
> or just show the answer. It is the branch at the heart of every agent.
>
> The `tool_use_id` has to be echoed back exactly. It is how the model matches your result to the
> request it made. Get it wrong and the model behaves as if you ignored it.
>
> The assistant's own turn goes back into the history unchanged (`first.content`). Drop it and the
> conversation stops making sense.
>
> **What is still missing** is everything that makes this safe: checking that the tool name is one you
> actually have, validating the arguments before running anything, and capping how many times the loop
> may go round. That is Week 5, and now you know exactly which lines those checks wrap.

---

# Lab 6 : Does your prompt actually work?

**Deck section 7.**

You change a prompt. You try it on two or three questions. The answers look good. Ship it?

That is a **vibe check**, and it does not survive contact with a hundred real users. A prompt can look
better on three examples and be worse on the other ninety-seven.

What teams do instead is keep an **eval set**: a fixed list of questions where somebody has already
written down the right answer. Think of it as a test paper for your application.

You are going to run one by hand. Four tickets, two prompts, and you keep score yourself.

In [ ]:
PROMPT_A = "Categorise the customer's ticket."

PROMPT_B = """You are a support ticket router.

Classify the ticket into EXACTLY ONE of these categories:
- billing: charges, refunds, invoices, money owed
- delivery: shipping, tracking, late or missing parcels
- technical: app or website errors, crashes, buttons that do not work
- other: greetings, and anything not about our service

Rules:
- Decide on what the customer wants fixed, not on how angry they sound.
- If the customer wants money back, it is billing.

Reply with the category word only."""

print("Two prompts ready. Prompt A is vague. Prompt B is specific.")

### Three tickets, two prompts

Each cell below runs the same ticket through both prompts and prints the answer a human agreed was
correct. Read all three, then keep score at the end.

In [ ]:
# Ticket 1. The correct answer is: billing
TICKET = "You billed me for a plan I cancelled."

answer_a = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": PROMPT_A}, {"role": "user", "content": TICKET}],
    options={"temperature": 0},
)

answer_b = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": PROMPT_B}, {"role": "user", "content": TICKET}],
    options={"temperature": 0},
)

print("Expected :", "billing")
print("Prompt A :", answer_a["message"]["content"].strip())
print("Prompt B :", answer_b["message"]["content"].strip())

In [ ]:
# Ticket 2. The correct answer is: delivery
TICKET = "Tracking says delivered but nothing arrived."

answer_a = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": PROMPT_A}, {"role": "user", "content": TICKET}],
    options={"temperature": 0},
)

answer_b = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": PROMPT_B}, {"role": "user", "content": TICKET}],
    options={"temperature": 0},
)

print("Expected :", "delivery")
print("Prompt A :", answer_a["message"]["content"].strip())
print("Prompt B :", answer_b["message"]["content"].strip())

In [ ]:
# Ticket 3. The correct answer is: billing
# A trap: the customer insults the website, but what they want is their money back.
TICKET = "Your website is a disgrace and I want my money back."

answer_a = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": PROMPT_A}, {"role": "user", "content": TICKET}],
    options={"temperature": 0},
)

answer_b = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": PROMPT_B}, {"role": "user", "content": TICKET}],
    options={"temperature": 0},
)

print("Expected :", "billing")
print("Prompt A :", answer_a["message"]["content"].strip())
print("Prompt B :", answer_b["message"]["content"].strip())

### Keep score

Fill this in from what you just saw. One mark per correct answer.

| Ticket | Expected | Prompt A right? | Prompt B right? |
|---|---|---|---|
| You billed me for a plan I cancelled | billing | | |
| Tracking says delivered but nothing arrived | delivery | | |
| Your website is a disgrace and I want my money back | billing | | |
| **Total out of 3** | | | |

> **You just ran an eval.** That is all one is: fixed inputs, known answers, count the hits.
>
> **Two things to notice.**
>
> Prompt A often answers with a whole sentence rather than one word, so you have to squint to decide
> whether it was even right. Prompt B is easier to score *because* it was told to reply with one word.
> A prompt that is easy to measure is a prompt you can improve.
>
> And three tickets is a very small test. One lucky answer moves the score by a third. Real teams use
> dozens or hundreds of cases, and they add a new one every time they find a bug in production.
>
> **The loop this gives you:** change the prompt, run it again, compare the numbers, keep it or throw
> it away. Exactly like a test suite, and for exactly the same reason. Doing that by hand does not
> scale, which is why the next step is a few lines of code that run all the cases for you, and why
> prompts belong in version control alongside the code that uses them.

---

## Recap

| You learned | The line to remember |
|---|---|
| Message roles | A conversation is a list of messages, each with a role. |
| No memory | The model remembers nothing. You re-send the history, and you pay for it every time. |
| System prompts | Role, constraints, output format, examples, escape hatch. The last one stops invented answers. |
| Few-shot | Your examples are not decoration. They are the decision boundary. |
| Chain-of-thought | Buys accuracy with tokens. Not worth it for simple questions. |
| Self-critique | One call is cheap and invisible. Two calls give you a verdict you can act on. |
| Structured outputs | Describe the shape, generate against it, **validate before you use it**. |
| Function calling | The model decides. Your code does the work. |
| Evals | A prompt change should produce a number, not a feeling. |